# Load text and processed text files

In [1]:
import pandas as pd
import numpy as np
import nltk
from nltk.corpus import stopwords
from nltk import word_tokenize, FreqDist
from nltk.stem.wordnet import WordNetLemmatizer
import string
import plotly
import plotly.graph_objects as go
import plotly.express as px
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer

ModuleNotFoundError: No module named 'nltk'

In [ ]:
file = open('../data/credit_reporting_text.txt')
credit_reporting_text = file.read()
file.close()

file = open('../data/debt_collection_text.txt')
debt_collection_text = file.read()
file.close()

file = open('../data/mortgages_and_loans_text.txt')
mortgages_and_loans_text = file.read()
file.close()

file = open('../data/credit_card_text.txt')
credit_card_text = file.read()
file.close()

file = open('../data/retail_banking_text.txt')
retail_banking_text = file.read()
file.close()

In [ ]:
df = pd.read_csv('../data/credit_reporting_text_processed.csv')
credit_reporting_text_processed = df['0'].tolist()

df = pd.read_csv('../data/debt_collection_text_processed.csv')
debt_collection_text_processed = df['0'].tolist()

df = pd.read_csv('../data/mortgages_and_loans_text_processed.csv')
mortgages_and_loans_text_processed = df['0'].tolist()

df = pd.read_csv('../data/credit_card_text_processed.csv')
credit_card_text_processed = df['0'].tolist()

df = pd.read_csv('../data/retail_banking_text_processed.csv')
retail_banking_text_processed = df['0'].tolist()

df = None

# Graph Most Frequent Words

In [ ]:
data = FreqDist(credit_reporting_text_processed).most_common(35)
fig = go.Figure()
fig.add_trace(go.Bar(x= [data[i][0] for i in range(len(data))], y=[data[i][1] for i in range(len(data))]))
fig.update_layout(title = "Most Common Words: Credit Reporting")
fig.show()

In [ ]:
data = FreqDist(debt_collection_text_processed).most_common(35)
fig = go.Figure()
fig.add_trace(go.Bar(x= [data[i][0] for i in range(len(data))], y=[data[i][1] for i in range(len(data))]))
fig.update_traces(marker_color='moccasin')
fig.update_layout(title = "Most Common Words: Debt Collection")
fig.show()

In [ ]:
data = FreqDist(mortgages_and_loans_text_processed).most_common(35)
fig = go.Figure()
fig.add_trace(go.Bar(x= [data[i][0] for i in range(len(data))], y=[data[i][1] for i in range(len(data))]))
fig.update_traces(marker_color='#1f77b4')
fig.update_layout(title = "Most Common Words: Mortgages and Loans")
fig.show()

In [ ]:
data = FreqDist(credit_card_text_processed).most_common(35)
fig = go.Figure()
fig.add_trace(go.Bar(x= [data[i][0] for i in range(len(data))], y=[data[i][1] for i in range(len(data))]))
fig.update_traces(marker_color='yellowgreen')
fig.update_layout(title = "Most Common Words: Credit Card")
fig.show()

In [ ]:
data = FreqDist(retail_banking_text_processed).most_common(35)
fig = go.Figure()
fig.add_trace(go.Bar(x= [data[i][0] for i in range(len(data))], y=[data[i][1] for i in range(len(data))]))
fig.update_traces(marker_color='darkred')
fig.update_layout(title = "Most Common Words: Retail Banking")
fig.show()

# TF-IDF

## Trying out TF-IDF

In [ ]:
# function to concat processed text as a single string inside a list
def concat_words(list):
    # remove any NaN's
    list = [i for i in list if i is not np.nan]

    # concat words
    concat_words = ''
    for word in list:
        concat_words += word + ' '
    return [concat_words]

### Trying out retail banking text

In [ ]:
retail_banking_corpus = concat_words(retail_banking_text_processed)

In [ ]:
vectorizer = TfidfVectorizer()
X_retail_banking = vectorizer.fit_transform(retail_banking_corpus)

In [ ]:
X_retail_banking.shape

In [ ]:
feature_names = vectorizer.get_feature_names()

In [ ]:
X_retail_banking_dense = X_retail_banking.T.todense()

In [ ]:
X_retail_banking_dense.shape

In [ ]:
df = pd.DataFrame(X_retail_banking_dense, index=feature_names, columns=["score"])
df = df.sort_values(by="score", ascending=False)
df.iloc[0:30]

In [ ]:
FreqDist(retail_banking_text_processed).most_common(30)

It's exactly the same as my FreqDist. I guess I need to use it on multiple corpus.

### Combining retail banking and credit card texts

In [ ]:
credit_card_corpus = concat_words(credit_card_text_processed)

In [ ]:
combined_corpus = credit_card_corpus + retail_banking_corpus

In [ ]:
X_combined = vectorizer.fit_transform(combined_corpus)

In [ ]:
feature_names = vectorizer.get_feature_names()

In [ ]:
X_combined_dense = X_combined.T.todense()

In [ ]:
X_combined_dense.shape

In [ ]:
df = pd.DataFrame(X_combined_dense, index=feature_names, columns=["credit_card", "retail"])
df = df.sort_values(by="retail", ascending=False)
df.iloc[0:30]

In [ ]:
df = df.sort_values(by="credit_card", ascending=False)
df.iloc[0:30]

## Running TF-IDF on all text

### Make full corpus

In [ ]:
# already did retail_banking and credit_card
credit_reporting_corpus = concat_words(credit_reporting_text_processed)
debt_collection_corpus = concat_words(debt_collection_text_processed)
mortgages_and_loans_corpus = concat_words(mortgages_and_loans_text_processed)

In [ ]:
full_corpus = (credit_reporting_corpus + debt_collection_corpus + mortgages_and_loans_corpus + 
               credit_card_corpus + retail_banking_corpus)

### Create TF-IDF Matrix

In [ ]:
vectorizer = TfidfVectorizer()
X_full_corpus = vectorizer.fit_transform(full_corpus)
feature_names = vectorizer.get_feature_names()
X_full_corpus_dense = X_full_corpus.T.todense()

In [ ]:
X_full_corpus_dense.shape

#### Create dataframe 

In [ ]:
column_names = ["credit_reporting", "debt_collection", "mortgages_and_loans", 
                "credit_card", "retail_banking"]

In [ ]:
df = pd.DataFrame(X_full_corpus_dense, index=feature_names, columns=column_names)
df = df.sort_values(by="credit_reporting", ascending=False)
df.iloc[0:15]

In [ ]:
df = df.sort_values(by="debt_collection", ascending=False)


In [ ]:
df = df.sort_values(by="mortgages_and_loans", ascending=False)
df.iloc[0:15]

### Create TF-IDF with bigrams

In [ ]:
vectorizer = TfidfVectorizer(ngram_range=(1,2))
X_full_corpus = vectorizer.fit_transform(full_corpus)
feature_names = vectorizer.get_feature_names()
X_full_corpus_dense = X_full_corpus.T.todense()

In [ ]:
df = pd.DataFrame(X_full_corpus_dense, index=feature_names, columns=column_names)
df = df.sort_values(by="credit_reporting", ascending=False)
df.iloc[0:15]

In [ ]:
df = df.sort_values(by="mortgages_and_loans", ascending=False)
df.iloc[0:15]

# CountVectorizer

In [ ]:
# using bigram
cv = CountVectorizer(ngram_range=[1,2])
X_cv_bigram = cv.fit_transform(full_corpus)
feature_names = cv.get_feature_names()
X_cv_bigram_dense = X_cv_bigram.T.todense()

In [ ]:
df = pd.DataFrame(X_cv_bigram_dense, index=feature_names, columns=column_names)
df = df.sort_values(by="credit_reporting", ascending=False)
df.iloc[0:15]

In [ ]:
df = df.sort_values(by="debt_collection", ascending=False)
df.iloc[0:15]

# Lemmatize corpora

I notice that the vectors have a lot of words like "payment"/"payments" and "account"/"accounts" repeated as top words.

**Note**: Here I'm using the dataframe from CountVectorization. I think using the TF-IDF dataframe would be comparable for EDA purposes, if not exactly the same.

## Examine most frequent words from all classes

In [ ]:
# Create combined list of top words

top_words = []
for i in range(5):
    
    # sort df by column position
    df = df.sort_values(by=df.columns[i], ascending=False)
    
    # make a list out of the top 100 index values
    top_100 = df[df.columns[i]].index.values.tolist()[0:100]
    top_words += top_100

In [ ]:
# look at the top words with duplicates removed 
top_words_set = set(top_words)
print(len(top_words_set))
top_words_set

## Perform lemmatization

In [ ]:
lemm = WordNetLemmatizer()

In [ ]:
def make_lemma(list_of_words):
    # remove any NaN's
    list_of_words = [i for i in list_of_words if i is not np.nan]
    
    list_to_return = []
    for idx, word in enumerate(list_of_words):
        list_to_return.append(lemm.lemmatize(word))
    return list_to_return

In [ ]:
credit_reporting_lemmatized = make_lemma(credit_reporting_text_processed)

In [ ]:
debt_collection_lemmatized = make_lemma(debt_collection_text_processed)
mortgages_and_loans_lemmatized = make_lemma(mortgages_and_loans_text_processed)
credit_card_lemmatized = make_lemma(credit_card_text_processed)
retail_banking_lemmatized = make_lemma(retail_banking_text_processed)

In [ ]:
# create a lemmatized full corpus

# full_corpus_lemmatized = ([credit_reporting_lemmatized] + [debt_collection_lemmatized] + 
#                           [mortgages_and_loans_lemmatized] + [credit_card_lemmatized] + 
#                           [retail_banking_lemmatized])

I don't have `full_corpus_lemmatized` in the correct structure for TF-IDF. It needs to be a list of five items, and each item is a single string with spaces between the words.

In [ ]:
credit_reporting_lemmatized = concat_words(credit_reporting_lemmatized)
debt_collection_lemmatized = concat_words(debt_collection_lemmatized)
mortgages_and_loans_lemmatized = concat_words(mortgages_and_loans_lemmatized)
credit_card_lemmatized = concat_words(credit_card_lemmatized)
retail_banking_lemmatized = concat_words(retail_banking_lemmatized)

In [ ]:
# re-create another lemmatized full corpus
full_corpus_lemmatized = (credit_reporting_lemmatized + debt_collection_lemmatized + 
                          mortgages_and_loans_lemmatized + credit_card_lemmatized + 
                          retail_banking_lemmatized)

# TF-IDF on lemmatized corpus

In [ ]:
vectorizer = TfidfVectorizer(ngram_range=(1,2))
X_lemmatized_corpus = vectorizer.fit_transform(full_corpus_lemmatized)
feature_names = vectorizer.get_feature_names()
X_lemmatized_corpus_dense = X_lemmatized_corpus.T.todense()

In [ ]:
df = pd.DataFrame(X_lemmatized_corpus_dense, index=feature_names, columns=column_names)
df = df.sort_values(by="credit_reporting", ascending=False)
df.iloc[0:5]

In [ ]:
df = df.sort_values(by="debt_collection", ascending=False)
df.iloc[0:5]

In [ ]:
df = df.sort_values(by="mortgages_and_loans", ascending=False)
df.iloc[0:5]

In [ ]:
df = df.sort_values(by="credit_card", ascending=False)
df.iloc[0:5]

In [ ]:
df = df.sort_values(by="retail_banking", ascending=False)
df.iloc[0:5]

## Pie graphs of top words

In [ ]:
df = df.sort_values(by="credit_reporting", ascending=False)
fig = px.pie(df, values=df.loc['credit'], names=column_names)
fig.update_layout(title = "'Credit': Top Term in Credit Reporting Compared to Other Classes")
fig.show()

In [ ]:
fig = px.pie(df, values=df.loc['debt'], names=column_names)
fig.update_layout(title = "'Debt': Top Term in Debt Collection Compared to Other Classes")
fig.show()

In [ ]:
fig = px.pie(df, values=df.loc['payment'], names=column_names)
fig.update_layout(title = "'Payment': Top Term in Mortgages and Loans compared to Other Classes")
fig.show()

In [ ]:
fig = px.pie(df, values=df.loc['card'], names=column_names)
fig.update_layout(title = "'Card': Top Term in Credit Card compared to Other Classes")
fig.show()

In [ ]:
fig = px.pie(df, values=df.loc['account'], names=column_names)
fig.update_layout(title = "'Account': Top Term in Retail Banking compared to Other Classes")
fig.show()

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
df.tail()

Export just the top 1500 rows for use in post-modeling EDA.

In [ ]:
df_export = df.iloc[:1500]

In [ ]:
df_export.shape

In [ ]:
df_export.to_csv('../project_data/corpus_lemm_tfidf.csv')